# dataclass-training-args — worked example 3: Build a dataclass from a noisy dict using dataclasses.fields

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataclass-training-args`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`dataclasses.fields(Cls)` returns metadata for every declared field, so you can introspect a dataclass's schema at runtime. A common config pattern is to take a dict that may contain extra/unknown keys (e.g. parsed from JSON or argparse) and keep only the keys that match real fields before constructing the instance.

## Worked solution

**Goal:** write a constructor that filters a messy dict down to a dataclass's known fields, then validates the result.

**Step 1 — declare the schema.** `OptimArgs` has `lr`, `weight_decay`, and `momentum` with defaults. These declarations are the single source of truth for what keys are legal.

**Step 2 — discover the legal field names.** `{f.name for f in fields(OptimArgs)}` builds the set `{'lr', 'weight_decay', 'momentum'}` directly from the dataclass, so the filter stays in sync if we add a field later — no hard-coded key list to forget to update.

**Step 3 — filter the incoming dict.** We comprehension over the raw dict keeping only `k` that are in the legal set. The stray key `'gpu_id'` is silently dropped, which is what lets us feed in a superset of config without `TypeError: unexpected keyword argument`.

**Step 4 — construct and validate.** We splat the filtered dict into `OptimArgs(**clean)`. Its `__post_init__` checks that `momentum` is in `[0, 1)` and `weight_decay >= 0`, so a bad numeric value still fails loudly even though unknown keys were tolerated.

**Why it works:** introspecting `fields()` separates the two concerns — *which keys are structurally valid* (driven by the schema) versus *which values are semantically valid* (driven by `__post_init__`). The result is a robust loader that accepts noisy input but never silently accepts a bad number.

In [ ]:
from dataclasses import dataclass, fields

@dataclass
class OptimArgs:
    lr: float = 1e-2
    weight_decay: float = 0.0
    momentum: float = 0.9

    def __post_init__(self):
        if not (0.0 <= self.momentum < 1.0):
            raise ValueError(f'momentum must be in [0, 1), got {self.momentum}')
        if self.weight_decay < 0:
            raise ValueError(f'weight_decay must be >= 0, got {self.weight_decay}')

def from_dict(raw):
    legal = {f.name for f in fields(OptimArgs)}
    clean = {k: v for k, v in raw.items() if k in legal}
    return OptimArgs(**clean)

args = from_dict({'lr': 5e-3, 'momentum': 0.95, 'gpu_id': 3})
print(args.lr, args.momentum, args.weight_decay)
print(hasattr(args, 'gpu_id'))